In [ ]:
# =====================================================
# 1. 라이브러리
# =====================================================

import re
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from collections import Counter
from wordcloud import WordCloud

In [ ]:
# =====================================================
# 한글 폰트 설정: Matplotlib + WordCloud 공통
# =====================================================

def set_korean_font():
    font_candidates = [
        "Malgun Gothic",      # Windows
        "AppleGothic",        # Mac
        "NanumGothic",        # Linux / Colab
        "Noto Sans CJK KR"
    ]

    installed_fonts = fm.fontManager.ttflist

    for font in installed_fonts:
        if font.name in font_candidates:
            plt.rcParams["font.family"] = font.name
            plt.rcParams["axes.unicode_minus"] = False

            print(f"✅ Matplotlib 한글 폰트 설정 완료: {font.name}")
            print(f"✅ WordCloud font_path: {font.fname}")

            return font.fname

    print("⚠️ 자동으로 한글 폰트를 찾지 못했습니다.")
    return None


font_path = set_korean_font()

In [ ]:
# =====================================================
# 2. 데이터 불러오기
# =====================================================

INPUT_FILE = "liverpool_youtube_cafe_final_labeled.xlsx"

df = pd.read_excel(INPUT_FILE)

print(df.shape)
print(df.columns.tolist())

# 카페 데이터만 사용
cafe_df = df[df["source"] == "cafe"].copy()

# 38R 전/후 확인
print(cafe_df["period"].value_counts(dropna=False))

display(cafe_df[["period", "title", "content_type", "text_clean"]].head())

In [ ]:
# =====================================================
# 3. 한글 폰트 경로 찾기
# =====================================================

def get_korean_font_path():
    font_candidates = [
        "Malgun Gothic",      # Windows
        "AppleGothic",        # Mac
        "NanumGothic",        # Linux / Colab
        "Noto Sans CJK KR"
    ]

    for font in fm.fontManager.ttflist:
        if font.name in font_candidates:
            print(f"✅ WordCloud 한글 폰트 설정: {font.name}")
            return font.fname

    print("⚠️ 한글 폰트를 찾지 못했습니다. font_path를 직접 지정해야 할 수 있습니다.")
    return None

font_path = get_korean_font_path()

In [ ]:
# =====================================================
# 4. 텍스트 전처리 / 토큰화
# =====================================================

# 기본 불용어
stopwords = set([
    # [1] 기존 & 유튜브/커뮤니티 은어
    '진짜', '너무', '영상', '오늘', '경기', '그냥', '진심', 'ㅋㅋ', 'ㅎㅎ', 'ㅠㅠ', 'ㅜㅜ', '댓글', '개인',
    '이거', '저거', '그거', '뭔가', '정말', '지금', '다시', '계속', '가장', '제일', '조금', '정도','개추',
    
    # [2] 축구 일반 용어 (분석 인사이트가 없는 뻔한 단어)
    '축구', '하이라이트', '감독', '선수', '시즌', '리그', '전반', '후반', '골', '볼', '팬', '응원', '생각', '사람', '우리', '이번',
    
    # [3] 리버풀 지칭
    '리버풀', '리중딱', '전범풀',
    
    # [4] 타 팀 및 관련 선수 (노이즈 제거)
    '맨유', '맨체스터유나이티드', '맹구', '맨시티', '맨체스터시티', 
    '아스날', '아스널', '개스널', '리버풀', '전범풀', '첼시', '첼강딱', 
    '뉴캐슬', '빌라', '에버튼', '울버햄튼', '황희찬', 
    '레알', '바르샤', '뮌헨', '김민재', '이강인', '파리', 'psg', '리즈',
    '웨스트햄', '아스톤빌라','웨햄','클럽',
    
    
    # [6] 형태소 분석(Okt) 후 흔히 남는 무의미한 두 글자 명사들
    '때문', '위해', '대한', '관련', '진짜' ,"작성","짧은","글", "질문", "삭제", "필수", "검색", "경기시작", "익일", "오전" ,"일반","일반글", "작성", "금지","게시","필독"
])

# 축구 분석에서 너무 일반적인 단어는 필요에 따라 제거
football_common_stops = set([
    "리그", "프리미어", "라운드", "승리", "패배", "무승부",
    "전반", "후반", "골", "득점", "실점", "수비", "공격",
    "라인업", "교체", "전술"
])

# 너무 많이 빠지면 아래 줄 주석 처리
stopwords = stopwords | football_common_stops


def basic_clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Kiwi가 있으면 명사/고유명사 중심 토큰화, 없으면 간단 토큰화
try:
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    USE_KIWI = True
    print("✅ Kiwi 토큰화 사용")
except:
    USE_KIWI = False
    print("⚠️ Kiwi 없음: 기본 정규식 토큰화 사용")


def tokenize_korean(text):
    text = basic_clean_text(text)

    if USE_KIWI:
        tokens = []

        for token in kiwi.tokenize(text):
            # NNG 일반명사, NNP 고유명사, SL 외국어
            if token.tag in ["NNG", "NNP", "SL"]:
                word = token.form.strip()

                if len(word) >= 2 and word not in stopwords:
                    tokens.append(word)

        return tokens

    else:
        tokens = re.findall(r"[가-힣a-zA-Z]{2,}", text)

        tokens = [
            t for t in tokens
            if t not in stopwords
        ]

        return tokens

In [ ]:
# =====================================================
# 5. 38R 이전 / 이후 텍스트 토큰 만들기
# =====================================================

before_df = cafe_df[cafe_df["period"] == "before_38R"].copy()
after_df = cafe_df[cafe_df["period"] == "after_38R"].copy()

print("38R 이전 데이터 수:", before_df.shape)
print("38R 이후 데이터 수:", after_df.shape)


def make_tokens_from_df(data):
    all_tokens = []

    for text in data["text_clean"].dropna():
        all_tokens.extend(tokenize_korean(text))

    return all_tokens


before_tokens = make_tokens_from_df(before_df)
after_tokens = make_tokens_from_df(after_df)

before_counter = Counter(before_tokens)
after_counter = Counter(after_tokens)

print("38R 이전 토큰 수:", len(before_tokens))
print("38R 이후 토큰 수:", len(after_tokens))

print("\n38R 이전 상위 단어")
print(before_counter.most_common(20))

print("\n38R 이후 상위 단어")
print(after_counter.most_common(20))

In [ ]:
# =====================================================
# 6. 워드클라우드 생성 함수
# =====================================================

def plot_wordcloud(counter, title, filename):
    wc = WordCloud(
        font_path=font_path,
        width=1200,
        height=700,
        background_color="white",
        max_words=150,
        prefer_horizontal=0.9,
        random_state=42
    ).generate_from_frequencies(counter)

    plt.figure(figsize=(12, 7))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title, fontsize=18, fontweight="bold", pad=18)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# =====================================================
# 7. 38R 이전 워드클라우드
# =====================================================

plot_wordcloud(
    before_counter,
    "38R 이전 팬 카페 워드클라우드",
    "wordcloud_cafe_before_38R.png"
)

In [ ]:
# =====================================================
# 8. 38R 이후 워드클라우드
# =====================================================

plot_wordcloud(
    after_counter,
    "38R 이후 팬 카페 워드클라우드",
    "wordcloud_cafe_after_38R.png"
)

In [ ]:
# =====================================================
# 9. 상위 단어 막대그래프 함수
# =====================================================

def plot_top_words(counter, title, filename, top_n=20):
    top_words = counter.most_common(top_n)

    word_df = pd.DataFrame(top_words, columns=["word", "count"])
    word_df = word_df.sort_values("count", ascending=True)

    plt.figure(figsize=(9, 7))
    plt.barh(word_df["word"], word_df["count"])
    plt.title(title, fontsize=16, fontweight="bold", pad=15)
    plt.xlabel("빈도")
    plt.ylabel("단어")
    plt.grid(axis="x", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# =====================================================
# 10. 38R 이전 상위 단어
# =====================================================

plot_top_words(
    before_counter,
    "38R 이전 팬 카페 상위 단어",
    "top_words_cafe_before_38R.png",
    top_n=20
)

In [ ]:
# =====================================================
# 11. 38R 이후 상위 단어
# =====================================================

plot_top_words(
    after_counter,
    "38R 이후 팬 카페 상위 단어",
    "top_words_cafe_after_38R.png",
    top_n=20
)